In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import scanpy as sc
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import umap
from PIL import Image
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from torchvision.models.vision_transformer import vit_b_16


In [ ]:

# ------------------------
# Dataset
# ------------------------
class XeniumCellDataset(Dataset):
    def __init__(self, gene_expr_dir, image_dir, transform=None):
        self.gene_expr_dir = gene_expr_dir
        self.image_dir = image_dir
        self.transform = transform or transforms.ToTensor()
        self.data_pairs = []
        self.cell_types = []
        self.batches = []
        self.shared_genes = None
        self._prepare_data()

    def _prepare_data(self):
        expr_files = [f for f in os.listdir(self.gene_expr_dir) if f.endswith('.h5ad')]
        shared_genes = None
        temp_storage = []

        for fname in tqdm(expr_files, desc="Finding shared genes"):
            adata = sc.read_h5ad(os.path.join(self.gene_expr_dir, fname))
            genes = set(adata.var_names)
            shared_genes = genes if shared_genes is None else shared_genes & genes
            temp_storage.append((fname, adata))

        self.shared_genes = sorted(list(shared_genes))
        print(f"✅ Shared genes found: {len(self.shared_genes)}")

        for fname, adata in tqdm(temp_storage, desc="Collecting image-gene pairs"):
            img_folder = os.path.join(self.image_dir, fname.replace(".h5ad", ""))
            if not os.path.isdir(img_folder):
                continue
            adata = adata[:, self.shared_genes]
            for cell_id in adata.obs.index:
                img_path = os.path.join(img_folder, f"{cell_id}.png")
                if os.path.exists(img_path):
                    expr_vector = adata[cell_id].X.toarray().flatten() if hasattr(adata[cell_id].X, "toarray") else adata[cell_id].X.flatten()
                    self.data_pairs.append((img_path, expr_vector))
                    self.cell_types.append(adata.obs[cell_id]["cell_type"] if "cell_type" in adata.obs.columns else "Unknown")
                    self.batches.append(fname.replace(".h5ad", ""))

    def __len__(self):
        return len(self.data_pairs)

    def __getitem__(self, idx):
        img_path, expr_vector = self.data_pairs[idx]
        image = Image.open(img_path).convert('RGB')
        image = self.transform(image)
        expr_tensor = torch.tensor(expr_vector, dtype=torch.float32)
        return image, expr_tensor

# ------------------------
# Model
# ------------------------
class Image2Transcripts(nn.Module):
    def __init__(self, num_genes, embed_dim=768):
        super().__init__()
        self.image_encoder = vit_b_16(pretrained=True)
        self.image_encoder.heads = nn.Identity()
        self.gene_encoder = nn.Sequential(
            nn.Linear(num_genes, 512),
            nn.ReLU(),
            nn.Linear(512, embed_dim),
        )

    def forward(self, image, gene):
        image_embed = self.image_encoder(image)
        gene_embed = self.gene_encoder(gene)
        return image_embed, gene_embed

# ------------------------
# Evaluation
# ------------------------
def compute_topk_accuracy(image_embed, gene_embed, topk=(1, 5)):
    image_embed = F.normalize(image_embed, dim=1)
    gene_embed = F.normalize(gene_embed , dim=1)
    sim_matrix = image_embed @ gene_embed.T
    results = {}
    for k in topk:
        topk_indices = torch.topk(sim_matrix, k=k, dim=1).indices
        match = torch.arange(sim_matrix.size(0), device=sim_matrix.device).unsqueeze(1)
        correct = (topk_indices == match).any(dim=1).float()
        acc = correct.mean().item()
        results[f"top{k}_acc"] = acc
    return results, sim_matrix.cpu().numpy()

def plot_umap(image_embed, gene_embed, cell_types, batches, tag="val"):
    image_embed = F.normalize(image_embed, dim=1).cpu().numpy()
    gene_embed  = F.normalize(gene_embed , dim=1).cpu().numpy()
    all_embed = np.concatenate([image_embed, gene_embed], axis=0)
    labels = ["Image"] * len(image_embed) + ["Gene"] * len(gene_embed)

    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, metric="cosine", random_state=42)
    embedding = reducer.fit_transform(all_embed)

    df = pd.DataFrame({
        "UMAP1": embedding[:, 0],
        "UMAP2": embedding[:, 1],
        "Modality": labels,
        "CellType": cell_types + cell_types,
        "Batch": batches + batches
    })

    for col, palette in [("Modality", None), ("CellType", "tab20"), ("Batch", "Set2")]:
        plt.figure(figsize=(6, 5))
        sns.scatterplot(data=df, x="UMAP1", y="UMAP2", hue=col, alpha=0.6, s=20, palette=palette)
        plt.title(f"{tag} - {col}")
        plt.tight_layout()
        plt.savefig(f"output_188_batch/{tag}_umap_{col.lower()}.pdf")


In [ ]:

# ------------------------
# Run Validation
# ------------------------
if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"🖥️ Using device: {device}")

    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
    ])

    dataset = XeniumCellDataset("data/demo/gene_expression", "data/demo//images", transform)
    dataloader = DataLoader(dataset, batch_size=32, shuffle=False, num_workers=8)

    model = Image2Transcripts(num_genes=len(dataset.shared_genes)).to(device)
    model.load_state_dict(torch.load("output/best_model.pt"))
    model.eval()

    all_image_embed, all_gene_embed = [], []
    for imgs, genes in tqdm(dataloader, desc="Embedding"):
        imgs, genes = imgs.to(device), genes.to(device)
        img_e, gene_e = model(imgs, genes)
        all_image_embed.append(img_e)
        all_gene_embed.append(gene_e)

    all_image_embed = torch.cat(all_image_embed, dim=0)
    all_gene_embed  = torch.cat(all_gene_embed , dim=0)

    metrics, sim_matrix = compute_topk_accuracy(all_image_embed, all_gene_embed)
    print("🎯 Top-k Accuracy:", metrics)
    with open("output/validation_metrics.txt", "w") as f:
        for k, v in metrics.items():
            f.write(f"{k}: {v:.4f}\n")


🖥️ Using device: cuda


Finding shared genes: 100%|██████████| 1/1 [00:00<00:00,  8.56it/s]


✅ Shared genes found: 372


/archive/DPDS/Xiao_lab/shared/jia_yao/envs/image2transcripts/lib/python3.9/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/archive/DPDS/Xiao_lab/shared/jia_yao/envs/image2transcripts/lib/python3.9/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ViT_B_16_Weights.IMAGENET1K_V1`. You can also use `weights=ViT_B_16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Embedding:   2%|▏         | 8/493 [00:02<02:09,  3.73it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 74.00 MiB. GPU 0 has a total capacity of 31.73 GiB of which 74.44 MiB is free. Including non-PyTorch memory, this process has 31.48 GiB memory in use. Of the allocated memory 30.57 GiB is allocated by PyTorch, and 550.59 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models.vision_transformer import vit_b_16
from PIL import Image
import scanpy as sc
import pandas as pd
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import umap

# ------------------------
# Dataset
# ------------------------
class XeniumCellDataset(Dataset):
    def __init__(self, gene_expr_dir, image_dir, transform=None):
        self.gene_expr_dir = gene_expr_dir
        self.image_dir = image_dir
        self.transform = transform or transforms.ToTensor()
        self.data_pairs = []
        self.cell_types = []
        self.batches = []
        self.shared_genes = None
        self._prepare_data()

    def _prepare_data(self):
        expr_files = [f for f in os.listdir(self.gene_expr_dir) if f.endswith('.h5ad')]
        shared_genes = None
        temp_storage = []

        for fname in tqdm(expr_files, desc="Finding shared genes"):
            adata = sc.read_h5ad(os.path.join(self.gene_expr_dir, fname))
            genes = set(adata.var_names)
            shared_genes = genes if shared_genes is None else shared_genes & genes
            temp_storage.append((fname, adata))

        self.shared_genes = sorted(list(shared_genes))
        print(f"✅ Shared genes found: {len(self.shared_genes)}")

        for fname, adata in tqdm(temp_storage, desc="Collecting image-gene pairs"):
            img_folder = os.path.join(self.image_dir, fname.replace(".h5ad", ""))
            if not os.path.isdir(img_folder):
                continue
            adata = adata[:, self.shared_genes]
            for i, cell_id in enumerate(adata.obs.index):
                img_path = os.path.join(img_folder, f"{cell_id}.png")
                if os.path.exists(img_path):
                    try:
                        expr_vector = adata[cell_id].X.toarray().flatten() if hasattr(adata[cell_id].X, "toarray") else adata[cell_id].X.flatten()
                        self.data_pairs.append((img_path, expr_vector))
                        self.cell_types.append(str(adata.obs[cell_id]["cell_type"]) if "cell_type" in adata.obs.columns else "Unknown")
                        self.batches.append(fname.replace(".h5ad", ""))
                    except Exception as e:
                        print(f"❌ Error reading {cell_id} in {fname}: {e}")

    def __len__(self):
        return len(self.data_pairs)

    def __getitem__(self, idx):
        img_path, expr_vector = self.data_pairs[idx]
        try:
            image = Image.open(img_path).convert('RGB')
            image = self.transform(image)
        except Exception as e:
            print(f"[Image Error] {img_path}: {e}")
            image = torch.zeros(3, 224, 224)
        expr_tensor = torch.tensor(expr_vector, dtype=torch.float32)
        return image, expr_tensor

# ------------------------
# Model
# ------------------------
class Image2Transcripts(nn.Module):
    def __init__(self, num_genes, embed_dim=768):
        super().__init__()
        self.image_encoder = vit_b_16(pretrained=True)
        self.image_encoder.heads = nn.Identity()
        self.gene_encoder = nn.Sequential(
            nn.Linear(num_genes, 512),
            nn.ReLU(),
            nn.Linear(512, embed_dim),
        )

    def forward(self, image, gene):
        image_embed = self.image_encoder(image)
        gene_embed = self.gene_encoder(gene)
        return image_embed, gene_embed

# ------------------------
# Evaluation functions
# ------------------------
def compute_topk_accuracy(image_embed, gene_embed, topk=(1, 5)):
    image_embed = F.normalize(image_embed, dim=1)
    gene_embed = F.normalize(gene_embed , dim=1)
    sim_matrix = image_embed @ gene_embed.T
    results = {}
    for k in topk:
        topk_indices = torch.topk(sim_matrix, k=k, dim=1).indices
        match = torch.arange(sim_matrix.size(0), device=sim_matrix.device).unsqueeze(1)
        correct = (topk_indices == match).any(dim=1).float()
        acc = correct.mean().item()
        results[f"top{k}_acc"] = acc
    return results, sim_matrix.cpu().numpy()

def plot_umap(image_embed, gene_embed, cell_types, batches, tag="val"):
    image_embed = F.normalize(image_embed, dim=1).cpu().numpy()
    gene_embed  = F.normalize(gene_embed , dim=1).cpu().numpy()
    all_embed = np.concatenate([image_embed, gene_embed], axis=0)
    labels = ["Image"] * len(image_embed) + ["Gene"] * len(gene_embed)

    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, metric="cosine", random_state=42)
    embedding = reducer.fit_transform(all_embed)

    df = pd.DataFrame({
        "UMAP1": embedding[:, 0],
        "UMAP2": embedding[:, 1],
        "Modality": labels,
        "CellType": cell_types + cell_types,
        "Batch": batches + batches
    })

    for col, palette in [("Modality", None), ("CellType", "tab20"), ("Batch", "Set2")]:
        plt.figure(figsize=(6, 5))
        sns.scatterplot(data=df, x="UMAP1", y="UMAP2", hue=col, alpha=0.6, s=20, palette=palette)
        plt.title(f"{tag} - {col}")
        plt.tight_layout()
        plt.savefig(f"validation/{tag}_umap_{col.lower()}.pdf")
        plt.close()

# ------------------------
# Main
# ------------------------
if __name__ == "__main__":
    os.makedirs("validation", exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"🖥️ Using device: {device}")

    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
    ])

    dataset = XeniumCellDataset("data/demo/gene_expression", "data/demo/images", transform)
    dataloader = DataLoader(dataset, batch_size=8, shuffle=False, num_workers=0)

    model = Image2Transcripts(num_genes=len(dataset.shared_genes)).to(device)
    model.load_state_dict(torch.load("output_188/best_model.pt"))
    model.eval()

    all_image_embed, all_gene_embed = [], []

    for imgs, genes in tqdm(dataloader, desc="Embedding"):
        torch.cuda.empty_cache()  # release unused memory
        imgs = imgs.to(device, non_blocking=True)
        genes = genes.to(device, non_blocking=True)
        with torch.no_grad():
            img_e, gene_e = model(imgs, genes)
            all_image_embed.append(img_e)
            all_gene_embed.append(gene_e)

    all_image_embed = torch.cat(all_image_embed, dim=0)
    all_gene_embed  = torch.cat(all_gene_embed , dim=0)

    metrics, _ = compute_topk_accuracy(all_image_embed, all_gene_embed)
    print("🎯 Validation Accuracy:", metrics)

    with open("validation/validation_metrics.txt", "w") as f:
        for k, v in metrics.items():
            f.write(f"{k}: {v:.4f}\n")

    plot_umap(
        all_image_embed,
        all_gene_embed,
        dataset.cell_types,
        dataset.batches,
        tag="val"
    )

🖥️ Using device: cuda


Finding shared genes: 100%|██████████| 1/1 [00:00<00:00,  1.67it/s]


✅ Shared genes found: 372


/work/DPDS/s439765/envs/spatial_tcr/lib/python3.9/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/work/DPDS/s439765/envs/spatial_tcr/lib/python3.9/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ViT_B_16_Weights.IMAGENET1K_V1`. You can also use `weights=ViT_B_16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Embedding: 100%|██████████| 1972/1972 [07:26<00:00,  4.41it/s]


🎯 Validation Accuracy: {'top1_acc': 0.009891573339700699, 'top5_acc': 0.04146851599216461}


/work/DPDS/s439765/envs/spatial_tcr/lib/python3.9/site-packages/umap/umap_.py:1945: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(f"n_jobs value {self.n_jobs} overridden to 1 by setting random_state. Use no seed for parallelism.")
/work/DPDS/s439765/envs/spatial_tcr/lib/python3.9/site-packages/numba/np/ufunc/parallel.py:371: NumbaWarning: The TBB threading layer requires TBB version 2021 update 6 or later i.e., TBB_INTERFACE_VERSION >= 12060. Found TBB_INTERFACE_VERSION = 12050. The TBB threading layer is disabled.
  warnings.warn(problem)


In [2]:
import scanpy as sc

In [3]:
adata = sc.read_h5ad("data/gene_expression/processed/output-XETG00248__0010314__CA518C__20240411__220137.h5ad")

In [4]:
adata.obs

,x_centroid,y_centroid,transcript_counts,control_probe_counts,control_codeword_counts,unassigned_codeword_counts,deprecated_codeword_counts,total_counts,cell_area,nucleus_area
aaaaafgk-1,1303.531860,7.972855,192,0,0,0,0,192,147.254537,13.366250
aaaaaodb-1,1315.297241,27.608980,282,0,0,0,0,282,241.947196,57.800002
aaaacapf-1,1339.159424,25.309008,425,0,0,0,0,425,237.792821,76.359222
aaaacoom-1,1330.057129,21.746971,119,0,0,0,0,119,53.465002,16.165938
aaaadopg-1,1297.726562,22.615023,457,0,0,0,0,457,265.518760,71.482346
...,...,...,...,...,...,...,...,...,...,...
oikkegag-1,2475.497559,5084.912109,127,1,0,0,0,128,81.868284,65.612034
oikkhanb-1,2471.349609,5089.084473,61,0,0,0,0,61,42.988752,33.776876
oikkhkjh-1,2399.953369,5118.531250,172,0,0,0,0,172,94.286253,68.682659
oikkhlgg-1,2401.638428,5110.999512,208,0,0,0,0,208,100.924222,79.068597
